### Get Vulnerabilities from OSV

In [12]:
import os
import glob
import json
import collections.abc
import pandas as pd
import sys
import requests
from datetime import datetime

VUL_PATH = "vuls.json"
SFP_PATH = "sfp.csv"
API_BASE_URL = "http://localhost:8080"

In [13]:
# transform all vulnerabilities into a single json file
def data_transform(in_path, out_path):
    files = glob.iglob(in_path+"/*")
    result = list()
    for file in files:
        with open(file, 'r') as infile:
            result.append(json.load(infile))
    with open(out_path, 'w') as output_file:
        json.dump(result, output_file, indent=4)

if not os.path.exists(VUL_PATH):
    os.system("curl https://osv-vulnerabilities.storage.googleapis.com/crates.io/all.zip -o all.zip")
    os.system("unzip all.zip -d rust_vuls")
    data_transform("./rust_vuls", VUL_PATH)
    os.system("rm all.zip")
    os.system("rm -r rust_vuls")

### Get SFP mappings from cwe.mitre.org

In [14]:
if not os.path.exists(SFP_PATH):
    os.system("curl https://cwe.mitre.org/data/csv/888.csv.zip -o sfp.zip")
    os.system("unzip -q -p sfp.zip > sfp.csv")
    os.system("rm sfp.zip")

In [15]:
# Download https://static.crates.io/db-dump.tar.gz
# expect ~1 GB download
if not os.path.exists("crates_io_db"):
    os.system("wget https://static.crates.io/db-dump.tar.gz")
    os.system("tar -xzf db-dump.tar.gz")
    os.system("rm db-dump.tar.gz")
    # and extract its only directory as crates_io_db/
    os.system("mv 20*/ crates_io_db")

df_crates = pd.read_csv("crates_io_db/data/crates.csv")
df_categories = pd.read_csv("crates_io_db/data/categories.csv")
df_crates_categories = pd.read_csv("crates_io_db/data/crates_categories.csv")
df_versions = pd.read_csv("crates_io_db/data/versions.csv")

In [16]:
def get_severity_levels():
    """Fetch severity levels from API and create lookup dict"""
    response = requests.get(f"{API_BASE_URL}/severity-levels")
    response.raise_for_status()
    severity_levels = response.json()
    # API returns uppercase keys like "LOW", "HIGH"
    return {level['level'].upper(): level['id'] for level in severity_levels}

def get_vulnerability_types():
    """Fetch vulnerability types from API and create lookup dict"""
    response = requests.get(f"{API_BASE_URL}/vulnerability-types")
    response.raise_for_status()
    vuln_types = response.json()
    return {vtype['name']: vtype['id'] for vtype in vuln_types}

def get_or_create_package(package_name):
    """Get package ID from API, create if doesn't exist"""
    response = requests.get(f"{API_BASE_URL}/packages/{package_name}")
    if response.status_code == 200:
        return response.json()['id']
    elif response.status_code == 404:
        package_data = {"name": package_name}
        create_response = requests.post(f"{API_BASE_URL}/packages", json=package_data)
        create_response.raise_for_status()
        return create_response.json()['id']
    else:
        response.raise_for_status()

def create_vulnerability(vul_data):
    """Create vulnerability record via API"""
    response = requests.post(f"{API_BASE_URL}/vulnerabilities", json=vul_data)
    response.raise_for_status()
    return response.json()

### Format the vulnerabilities and merge vulnerability info

In [17]:
# Fetch lookup data from API
severity_lookup = get_severity_levels()
vuln_type_lookup = get_vulnerability_types()

df_cve = pd.read_json(VUL_PATH)
df_sfp = pd.read_csv(SFP_PATH)

In [18]:
def get_sfps_from_cwes():
    taxonomy = df_sfp["Affected Resources"].apply(lambda x: None if type(x)==float else (x.split("::TAXONOMY NAME:")))
    primary_cluster = {"SFP1":"Risky Values", "SFP2":"Unused Entities", "SFP3":"API", "SFP4":"Exception Management", "SFP5":"Exception Management", "SFP6":"Exception Management", 
                        "SFP7":"Memory Access", "SFP8":"Memory Access", "SFP9":"Memory Access", "SFP10":"Memory Access", "SFP11":"Memory Access", 
                        "SFP12":"Memory Management", "SFP13":"Resource Management", "SFP14":"Resource Management", "SFP15":"Resource Management", 
                        "SFP16":"Path Resolution", "SFP17":"Path Resolution", "SFP18":"Path Resolution", 
                        "SFP19":"Synchronization", "SFP20":"Synchronization", "SFP21":"Synchronization", "SFP22":"Synchronization",
                        "SFP23":"Information Leak", "SFP24":"Tainted Input", "SFP25":"Tainted Input", "SFP26":"Tainted Input", "SFP27":"Tainted Input", 
                        "SFP28":"Entry Points", "SFP29":"Authentication", "SFP30":"Authentication", "SFP31":"Authentication", "SFP32":"Authentication", "SFP33":"Authentication", "SFP34":"Authentication",
                        "SFP35":"Access Control", "SFP36":"Privilege", "SFP37":"Faulty Resource Release", "SFP38":"Failure to Release Memory"}
    def getSFP(x):
        if x is None:
            return x
        for t in x:
            if 'Software Fault Patterns' in t:
                sfp = t.split(':')[2]
                return primary_cluster[sfp] if sfp and 'SFP' in sfp else None
        return None
    taxonomy = taxonomy.apply(lambda x: getSFP(x) )
    taxonomy[908] = taxonomy[909] = "Exception Management"
    taxonomy[131] = taxonomy[787] = taxonomy[824]= taxonomy[119]= taxonomy[125] = "Memory Access"
    taxonomy[758] = "API"
    taxonomy[843] = "Risky Values"
    taxonomy[770] = taxonomy[772] = taxonomy[789] = "Resource Management"
    taxonomy[706] = "Path Resolution"
    taxonomy[362] = "Synchronization"
    taxonomy[668] = taxonomy[200] = taxonomy[203]= taxonomy[208]= taxonomy[377] = "Information Leak"
    taxonomy[129] = taxonomy[427] = taxonomy[172]= taxonomy[444]= taxonomy[198] = taxonomy[94] = taxonomy[351] = "Tainted Input"
    taxonomy[295] = "Authentication"
    taxonomy[279] = "Access Control"
    taxonomy[269] = "Privilege"
    taxonomy[327] = taxonomy[1240] = taxonomy[347] = "Cryptography"
    taxonomy[330] = taxonomy[338] = taxonomy[340] = "Predictability"
    taxonomy[657] = taxonomy[670] = taxonomy[682]= taxonomy[697]= taxonomy[188] = taxonomy[193] = taxonomy[835] = "Other"
    return taxonomy

In [19]:
from cvss import CVSS3, CVSS4

def cvss_version_autodetect(v):
    if not v:
        return None
    elif type(v) is str and str.lower(v) in ['low', 'medium', 'high', 'critical', 'moderate']:
        if str.lower(v) == 'moderate':
            return 'MEDIUM'
        else:
            return str.upper(v)
    elif type(v) is not str:
        raise TypeError()
    elif v.startswith('CVSS:3'):
        return CVSS3(v).severities()[0].upper()
    elif v.startswith('CVSS:4'):
        return CVSS4(v).severities()[0].upper()
    else:
        raise TypeError()

def get_vul_severity(data):
    temp = data.apply(lambda x:
        (
            x['affected'][0]['database_specific']['cvss']
            if 'cvss' in x['affected'][0]['database_specific']
            else (
                x['database_specific']['severity']
                if 'severity' in x['database_specific']
                else None
            )
        )
        if type(x['severity'])==float
        else x['severity'][0]['score']
        , axis=1)
    return temp.apply(cvss_version_autodetect)
    
def get_vul_package(data):
    return pd.Series(list(d[0]['package']['name'] for d in data['affected']))

def get_vul_repo_url(data):
    return data['package'].apply(lambda x: df_crates['repository'][df_crates['name']==x].values[0] if len(df_crates['repository'][df_crates['name']==x].values)!=0 else None)

def get_vul_sfp_id(data):
    temp = data.apply(lambda x:
        x['database_specific']['cwe_ids']
        if 'cwe_ids' in x.get('database_specific', {})
        else
            x['affected'][0]['database_specific'].get('categories', [])
        , axis=1)
    taxonomy = get_sfps_from_cwes()
    
    def get_sfp_from_cwe(x):
        cat2sfp = {"memory-exposure":"Memory Access", "memory-corruption":"Memory Management", "denial-of-service":"Resource Management", "file-disclosure":"Path Resolution",
                    "thread-safety":"Synchronization", "format-injection":"Tainted Input", "privilege-escalation":"Privilege", "crypto-failure":"Cryptography", "code-execution":"Other"
                  }
        res = list()
        for cwe in x:
            if "CWE" in cwe:
                id = int(cwe.split('-')[1])
                if id in taxonomy.index and taxonomy[id]:
                    res.append(str(taxonomy[id])) 
            else:
                res.append(str(cat2sfp[cwe]))
        return res

    return temp.apply(lambda x: str(get_sfp_from_cwe(x)))

def get_vul_version(data):
    def get_version(affects):
        versions = list()
        for affect in affects:
            if "ranges" in affect:
                introduced = affect["ranges"][0]["events"][0]["introduced"] if "introduced" in affect["ranges"][0]["events"][0] else None
                fixed = affect["ranges"][0]["events"][1]["fixed"] if len(affect["ranges"][0]["events"])>1 and "fixed" in affect["ranges"][0]["events"][1] else None
                versions.append((introduced, fixed))
        return str(versions)

    return data["affected"].apply(lambda x:get_version(x) )

def get_vul_reference(data):
    references_list = data.pop('references')
    return pd.Series(
        str(list(
            (
                ref['url']
                for ref in d
            )
            if isinstance(d, collections.abc.Iterable)
            else []))
        for d in references_list)

In [20]:
ordered_cve_columns = ['id', 'package', 'repo_url', 'sfp_id', 'modified', 'published', 'vul_version', 'summary', 'details', 'severity', 'references']
df_cve["severity"] = get_vul_severity(df_cve)
df_cve['package'] = get_vul_package(df_cve)
df_cve['repo_url'] = get_vul_repo_url(df_cve)
df_cve['sfp_id'] = get_vul_sfp_id(df_cve)
df_cve['vul_version'] = get_vul_version(df_cve)
df_cve['references'] = get_vul_reference(df_cve)
df_cve.drop(['aliases', 'database_specific', 'affected', 'schema_version', 'related'], axis=1, inplace=True)
df_cve = df_cve[ordered_cve_columns]
print(df_cve.head())
df_cve = df_cve.applymap(str)

### Remove duplicate vulnerabilities

In [21]:
import pandas as pd
import ast

df_res = list()
df_master = df_cve
packages = df_master.groupby('package').groups
print(len(df_master))

for key, values in packages.items():
    df_tmp = df_master.iloc[values]
    ref = ast.literal_eval(df_tmp.iloc[0]['references'])
    ref = list(filter(lambda tmp: "nvd" in tmp or "rustsec" in tmp, ref))
    references = [ref]
    vuls = [[0]]
    for i in range(1, len(values)):
        flag = True
        tmp = ast.literal_eval(df_tmp.iloc[i]['references'])
        tmp = list(filter(lambda ref: "nvd" in ref or "rustsec" in tmp, tmp))
        for idx, ref in enumerate(references):
            if set(ref) & set(tmp):
                references[idx].extend(tmp)
                vuls[idx].append(i)
                flag = False
        if flag:
            vuls.append([i])
            references.append(tmp)

    for vul_idx in vuls:
        vul = {
            'id': [],
            'package': df_tmp.iloc[vul_idx[0]]['package'],
            'repo_url': df_tmp.iloc[vul_idx[0]]['repo_url'],
            'sfp_id': [],
            'modified': df_tmp.iloc[vul_idx[0]]['modified'],
            'published': df_tmp.iloc[vul_idx[0]]['published'],
            'vul_version': [],
            'summary': "",
            'details': "",
            'severity': "",
            'references': []
        }
        for vvul in vul_idx:
            vul['id'].append(df_tmp.iloc[vvul]['id'])
            vul['published'] = min(vul['published'], df_tmp.iloc[vvul]['published'])
            vul['sfp_id'].extend(ast.literal_eval(df_tmp.iloc[vvul]['sfp_id']))
            vul['references'].extend(ast.literal_eval(df_tmp.iloc[vvul]['references']))
            vul['vul_version'].extend(ast.literal_eval(df_tmp.iloc[vvul]['vul_version']))
            vul['sfp_id'] = list(set(vul['sfp_id']))
            vul['references'] = list(set(vul['references']))
            vul['vul_version'] = list(set(vul['vul_version']))
            vul['summary'] = df_tmp.iloc[vvul]['summary'] + '\n' + vul['summary']
            vul['details'] = df_tmp.iloc[vvul]['details'] + '
' + vul['details']
            if df_tmp.iloc[vvul]['severity'] != 'nan':
                vul['severity'] = df_tmp.iloc[vvul]['severity']
        
        df_res.append(vul)
        
df_res = pd.DataFrame(df_res)
print(len(df_res))

In [22]:
# remove vulnerabilities that report unmaintained projects
df_res = df_res[~df_res['summary'].str.contains("unmaint", case=False)]
df_res = df_res[~df_res['summary'].str.contains("no longer maint", case=False)]
df_res = df_res[~df_res['summary'].str.contains("discontinue", case=False)]
print(f"Excluding unmaintained: {len(df_res)}")
# remove that report malicious code
df_res = df_res[~df_res['summary'].str.contains("malicious code", case=False)]
print(f"Excluding malicious crate deletions: {len(df_res)}")

### Send vulnerabilities to API

In [23]:
# Transform and send each vulnerability to API
for _, vul in df_res.iterrows():
    # Get package ID
    package_id = get_or_create_package(vul['package'])
    
    # Map severity to severity_id - use None (NULL) if no severity available instead of skipping
    severity_value = vul['severity']
    if severity_value and severity_value != 'nan':
        severity_id = severity_lookup.get(severity_value)
    else:
        severity_id = None  # NULL in database
    
    # Map SFP types to vulnerability_type_ids
    # Handle both string representations and actual lists
    sfp_value = vul['sfp_id']
    try:
        sfp_list = ast.literal_eval(sfp_value)
    except (ValueError, SyntaxError):
        if isinstance(sfp_value, list):
            sfp_list = sfp_value
        else:
            sfp_list = [sfp_value]
    
    type_ids = []
    for sfp in sfp_list:
        if sfp in vuln_type_lookup:
            type_ids.append(vuln_type_lookup[sfp])
    
    # Parse vulnerability IDs (GHSA, CVE, RUSTSEC) from vul['id'] list
    vulnerability_ids = []
    vul_id_list = vul['id'] if isinstance(vul['id'], list) else ast.literal_eval(str(vul['id']))
    for vid in vul_id_list:
        vid_str = str(vid)
        if vid_str.startswith('GHSA-'):
            vulnerability_ids.append({"id_type": "GHSA", "id_value": vid_str})
        elif vid_str.startswith('CVE-'):
            vulnerability_ids.append({"id_type": "CVE", "id_value": vid_str})
        elif vid_str.startswith('RUSTSEC-'):
            vulnerability_ids.append({"id_type": "RUSTSEC", "id_value": vid_str})
    
    # Parse affected versions from vul['vul_version']
    affected_versions = []
    vul_version_list = vul['vul_version'] if isinstance(vul['vul_version'], list) else ast.literal_eval(str(vul['vul_version']))
    for version_tuple in vul_version_list:
        if isinstance(version_tuple, (list, tuple)) and len(version_tuple) >= 2:
            intro, fixed = version_tuple[0], version_tuple[1]
            if intro or fixed:
                version_range = f">={intro or '0'},<{fixed}' if fixed else f">={intro or '0'}"
                affected_versions.append({
                    "version_range": version_range,
                    "introduced_version": intro,
                    "fixed_version": fixed
                })
    
    # Parse references from vul['references']
    references = []
    ref_list = vul['references'] if isinstance(vul['references'], list) else ast.literal_eval(str(vul['references']))
    for url in ref_list:
        if url and isinstance(url, str):
            references.append({"url": url})
    
    # Prepare vulnerability data with all fields
    vul_data = {
        "package_id": package_id,
        "severity_id": severity_id,
        "vulnerability_type_ids": type_ids if type_ids else [1],  # Default to first type if none found
        "summary": vul['summary'],
        "details": vul['details'],
        "published_at": vul['published'],
        "vulnerability_ids": vulnerability_ids,
        "affected_versions": affected_versions,
        "references": references
    }
    
    # Create vulnerability via API
    try:
        result = create_vulnerability(vul_data)
        print(f"Created vulnerability {vul['id']}: {result.get('id', 'unknown')}")
    except Exception as e:
        print(f"Failed to create vulnerability {vul['id']}: {e}")
        continue